In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_member = spark.sql(f"""
    SELECT 
         CAST(_c0 AS LONG) AS MBRSHP_SID
        ,_c1 AS MBRSHP_TYPE_ID
        -- remove leading zeros
        ,CAST(regexp_replace(trim(_c2), "^0+", "") AS DECIMAL(5,2)) AS MBRSHP_FEE_INC
        ,_c3 AS MBRSHP_SUB_TYPE
        ,CAST(_c4 AS DATE) AS MBRSHP_ENR_DT
        ,CAST(_c5 AS DATE) AS MBRSHP_EXP_DT
        ,CAST(_c6 AS DATE) AS MBRSHP_RNWL_DT
        ,_c7 AS RWDS_MBR_IND
    FROM 
        {bronze_master_member}
""").dropDuplicates()

df_member.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'member', config_validation, df_member, stats_etl_path
    )

### Merge

In [0]:
df_member.write.mode("overwrite").saveAsTable(silver_master_member)

if archive_flag:
    save_archive(df_member, silver_master_member_archive, run_as_date)